# Data Profiling — Urban Flow Analytics Taxi Dataset

**Run by:** [Praveen Madawalage]
**Date:** [2026/09/10]

Purpose: profile all 12 monthly CSVs before cleaning — check schema consistency,
dtypes, row counts, and date-range containment. This notebook's findings feed
directly into the cleaning decisions and the technical report's preprocessing section.

In [4]:
import duckdb
import glob
import pandas as pd

files = sorted(glob.glob("../data/raw/*.csv"))  # adjust path if running from notebooks/
print(f"Found {len(files)} files")
for f in files:
    print(" -", f)

Found 12 files
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
 - ../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


## 1. File Inventory & Schema Scan

Scanning each file's row count, column names, and dtypes using DuckDB
(lazy CSV scan — doesn't load full files into RAM).

In [5]:
con = duckdb.connect()
profile_rows = []

for f in files:
    schema = con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{f}')").fetchdf()
    row_count = con.execute(f"SELECT COUNT(*) AS n FROM read_csv_auto('{f}')").fetchone()[0]

    profile_rows.append({
        "file": f,
        "n_rows": row_count,
        "n_cols": len(schema),
        "columns": tuple(schema["column_name"].tolist()),
        "dtypes": tuple(schema["column_type"].tolist()),
    })
    print(f"{f}: {row_count:,} rows, {len(schema)} cols")

profile_df = pd.DataFrame(profile_rows)
profile_df  

../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv: 3,970,553 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv: 4,591,845 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv: 4,322,960 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv: 3,898,963 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv: 3,574,091 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv: 4,251,015 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv: 4,428,699 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv: 4,181,444 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv: 4,305,006 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv: 3,724,889 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv: 3,399,866 rows, 20 cols
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv: 3,952,451 rows, 20 cols


,file,n_rows,n_cols,columns,dtypes
0,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,3970553,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
1,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,4591845,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
2,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,4322960,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
3,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,3898963,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
4,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,3574091,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
5,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,4251015,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
6,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,4428699,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
7,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,4181444,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
8,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,4305006,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."
9,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...,3724889,20,"(provider_code, pickup_timestamp, dropoff_time...","(BIGINT, TIMESTAMP, TIMESTAMP, DOUBLE, DOUBLE,..."


In [6]:
# Save raw profile for the record / report appendix
profile_df.to_csv("../data/interim/file_schema_profile.csv", index=False)

## 2. Schema Consistency Check

- If unique column-name sets == 1 → all files match, safe to concatenate directly.
- If > 1 → inspect diffs below before merging (may be expected, e.g. a fee column
  introduced partway through the year per the data dictionary).

In [7]:
unique_schemas = profile_df["columns"].nunique()
unique_dtypes = profile_df["dtypes"].nunique()
print(f"Unique column-name sets across {len(files)} files: {unique_schemas}")
print(f"Unique dtype sets across {len(files)} files: {unique_dtypes}")

if unique_schemas > 1:
    print("\n⚠️ SCHEMA MISMATCH DETECTED. Columns differ between files:")
    baseline = set(profile_df["columns"].iloc[0])
    for i, row in profile_df.iterrows():
        diff = set(row["columns"]).symmetric_difference(baseline)
        if diff:
            print(f"  {row['file']}: differs by {diff}")
else:
    print("\n✅ All files share the same column set.")

if unique_dtypes > 1:
    print("\n⚠️ DTYPE MISMATCH DETECTED — inspect per-column below.")
else:
    print("✅ All files share the same dtypes.")

Unique column-name sets across 12 files: 1
Unique dtype sets across 12 files: 1

✅ All files share the same column set.
✅ All files share the same dtypes.


## 3. Date Range Containment Check

Each file should contain pickups almost entirely within its labeled month.
Rows falling well outside that range suggest mislabeled or corrupted data.

In [8]:
date_range_rows = []

for f in files:
    result = con.execute(f"""
        SELECT 
            MIN(pickup_timestamp) AS min_pickup, 
            MAX(pickup_timestamp) AS max_pickup,
            COUNT(*) AS total
        FROM read_csv_auto('{f}')
    """).fetchdf()
    row = result.to_dict("records")[0]
    row["file"] = f
    date_range_rows.append(row)
    print(f, row)

date_range_df = pd.DataFrame(date_range_rows)
date_range_df.to_csv("../data/interim/file_date_ranges.csv", index=False)
date_range_df

../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv {'min_pickup': Timestamp('2025-03-31 23:45:01'), 'max_pickup': Timestamp('2025-05-01 00:48:13'), 'total': 3970553, 'file': '../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv'}
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv {'min_pickup': Timestamp('2009-01-01 00:20:39'), 'max_pickup': Timestamp('2025-06-01 00:04:31'), 'total': 4591845, 'file': '../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv'}
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv {'min_pickup': Timestamp('2025-05-31 22:34:26'), 'max_pickup': Timestamp('2025-06-30 23:59:59'), 'total': 4322960, 'file': '../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv'}
../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv {'min_pickup': Timestamp('2009-01-01 08:52:26'), 'max_pickup': Timestamp('2025-07-31 23:59:59'), 'total': 3898963, 'file': '../data/raw/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv'}
../data/raw/Urban_Flow_Analytics

,min_pickup,max_pickup,total,file
0,2025-03-31 23:45:01,2025-05-01 00:48:13,3970553,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
1,2009-01-01 00:20:39,2025-06-01 00:04:31,4591845,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
2,2025-05-31 22:34:26,2025-06-30 23:59:59,4322960,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
3,2009-01-01 08:52:26,2025-07-31 23:59:59,3898963,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
4,2009-01-01 12:52:15,2025-09-01 00:00:29,3574091,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
5,2025-08-31 23:45:38,2025-10-01 00:00:11,4251015,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
6,2025-09-30 22:54:51,2025-11-01 00:32:12,4428699,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
7,2008-12-31 23:04:21,2025-11-30 23:59:59,4181444,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
8,2025-11-30 20:48:56,2025-12-31 23:59:59,4305006,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...
9,2025-12-31 23:57:29,2026-02-01 00:45:01,3724889,../data/raw/Urban_Flow_Analytics_Taxi_Dataset_...


### Date range findings

*(Fill in after reviewing output above)*

- [ ] File [x]: date range [min–max] — matches expected month? Y/N
- [ ] Any files with stray out-of-month rows? How many / what % of file?

## 4. Decisions Going Into Cleaning

Summarize the concrete actions this profiling leads to, before writing the cleaning notebook:

1. ...
2. ...
3. ...

*(This section doubles as source material for the "Data Preprocessing" section of the final technical report.)*